# Notebook 1 — M1: Preference Matching (embeddings + scoring híbrido)

**Real Estate Intelligence Platform (REIP)**

Construimos el conjunto de referencia sintético que usamos para evaluar Precision@k del modelo de
matching lifestyle-propiedad de M1, dado que no existe todavía ningún dato de comportamiento real
de usuarios. Este notebook cubre únicamente la construcción y verificación del conjunto de
referencia — la generación de embeddings y el cómputo de Precision@k se documentan por separado.


## Verificación de entorno y rutas

**Dataset de entrada:** `pipeline/data/processed/catalogo_residencial_limpio_6_2_1.csv` (Notebook
6.2.1). Excluimos `corregimiento == "zona_no_determinada"` (53 filas sin zona resuelta) y
`precio_no_evaluable == True` (14 filas con precio fuera de umbral de dominio) — un perfil de
lifestyle filtra por rango de precio, así que una fila con precio no evaluable no puede evaluarse
de forma confiable contra esos filtros.

**Zone Health:** `pipeline/data/processed/zone_health_composite_1_4_6.json` (Notebook 6.2.2), para
el desglose de 4 dimensiones por zona que usamos al definir qué corregimientos favorece cada
perfil de lifestyle. Zone Health se usa aquí como criterio de construcción del conjunto de
referencia — no como feature de ningún modelo de precio, consistente con la decisión ya cerrada de
mantenerlo fuera de KNN/RF.

**Modelo de embeddings:** confirmamos, contra la API real (no un nombre de modelo asumido), que
`gemini-embedding-001` está disponible para `embedContent` con la clave configurada, y que produce
vectores de dimensión 3072. `text-embedding-004` (nombre usado en versiones anteriores de la
documentación pública de Gemini) ya no existe en esta API — devuelve 404. Este notebook no genera
embeddings todavía; la verificación queda documentada aquí porque es la primera vez que este
notebook toca la API de Gemini.

**Variable de entorno:** cargamos `GEMINI_API_KEY` desde el `.env` de la raíz del repo (única
convención del proyecto, no se duplica un `.env` dentro de `notebooks/`) usando
`find_dotenv(usecwd=True)`, que busca hacia arriba en el árbol de directorios a partir del
directorio de trabajo — necesario porque el kernel de este notebook corre con `notebooks/` como
cwd, no la raíz del repo. Verificamos que la variable carga sin imprimir su valor en ninguna celda.

**Pesos del scorer híbrido (`CLAUDE.md`):** estructurado 0.6, semántico 0.4.


In [1]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv, find_dotenv
ruta_env = find_dotenv(usecwd=True)
assert ruta_env, "No se encontró .env en el árbol de directorios"
load_dotenv(ruta_env)

import os
assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY no está cargada"
print(".env cargado desde:", ruta_env)
print("GEMINI_API_KEY presente:", bool(os.environ.get("GEMINI_API_KEY")))


.env cargado desde: /Users/heliocastillo/Proyecto-REIP/Real-Estate-Intelligence-Platform/.env
GEMINI_API_KEY presente: True


In [2]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
modelos_embedding = [
    m.name for m in client.models.list() if "embedContent" in (m.supported_actions or [])
]
print("Modelos disponibles para embedContent:", modelos_embedding)

MODELO_EMBEDDING = "gemini-embedding-001"
resultado_prueba = client.models.embed_content(
    model=MODELO_EMBEDDING,
    contents="apartamento de 2 habitaciones en Bella Vista, cerca de parques y transporte",
)
DIMENSION_EMBEDDING = len(resultado_prueba.embeddings[0].values)
print(f"Modelo confirmado: {MODELO_EMBEDDING}, dimensión: {DIMENSION_EMBEDDING}")


Modelos disponibles para embedContent: ['models/gemini-embedding-001', 'models/gemini-embedding-2-preview', 'models/gemini-embedding-2']


Modelo confirmado: gemini-embedding-001, dimensión: 3072


## 1. Carga del catálogo y Zone Health

In [3]:
df_catalogo = pd.read_csv(REPO_ROOT / "pipeline" / "data" / "processed" / "catalogo_residencial_limpio_6_2_1.csv")

antes = len(df_catalogo)
df_catalogo = df_catalogo[df_catalogo["corregimiento"] != "zona_no_determinada"]
df_catalogo = df_catalogo[df_catalogo["precio_no_evaluable"] == False]
print(f"Catálogo filtrado para construcción de referencia: {len(df_catalogo)} / {antes} filas")

with open(REPO_ROOT / "pipeline" / "data" / "processed" / "zone_health_composite_1_4_6.json", encoding="utf-8") as f:
    zone_health = json.load(f)["zonas"]

desglose_zonas = {
    zona: registro["desglose"] for zona, registro in zone_health.items() if registro["desglose"] is not None
}
pd.DataFrame(desglose_zonas).T.round(3)


Catálogo filtrado para construcción de referencia: 1110 / 1177 filas


,seguridad,transporte,amenidades,walkability
Bella Vista,0.348,1.000,0.009,1.000
Betania,0.917,0.697,1.000,0.881
Parque Lefevre,0.000,0.523,0.513,0.752
Pedregal,0.305,0.000,0.000,0.000
San Francisco,1.000,0.677,0.159,0.788
El Cangrejo,0.348,1.000,0.009,1.000
Marbella,0.348,1.000,0.009,1.000
Obarrio,0.348,1.000,0.009,1.000


## 2. Perfiles de lifestyle sintéticos y reglas de ground truth

Definimos 6 perfiles con reglas estructuradas explícitas — no generadas por el modelo, no
inferidas de datos de usuario reales (no existen). Cada perfil combina: (a) un subconjunto de
corregimientos, elegido por qué dimensión de Zone Health prioriza el perfil, y (b) rangos
estructurados sobre `price_usd`, `bedrooms`, `area_m2`, tomados de los cuartiles reales del
catálogo (`price_usd`: p25=\$236,325, mediana=\$365,000, p75=\$650,000; `area_m2`: p25=80,
mediana=129, p75=245), no de umbrales arbitrarios.


In [4]:
PERFILES_LIFESTYLE = {
    "familia_con_ninos": {
        "descripcion": "Familia con niños que prioriza seguridad y cercanía a parques/colegios, necesita espacio.",
        "corregimientos": ["Betania", "San Francisco"],  # mayor (seguridad + amenidades) del desglose
        "price_usd_min": 150000,
        "price_usd_max": 700000,
        "bedrooms_min": 3,
        "area_m2_min": 120,
    },
    "profesional_joven": {
        "descripcion": "Profesional joven soltero que prioriza transporte y caminabilidad, unidad compacta.",
        "corregimientos": ["Bella Vista", "El Cangrejo", "Marbella", "Obarrio"],  # mayor (transporte + walkability)
        "price_usd_max": 450000,
        "bedrooms_min": 1,
        "bedrooms_max": 2,
    },
    "pareja_presupuesto_medio": {
        "descripcion": "Pareja sin hijos con presupuesto medio, busca balance general de calidad de zona.",
        "corregimientos": [z for z, r in zone_health.items() if r["composite"] is not None and r["composite"] >= 0.5],
        "price_usd_min": 236325,
        "price_usd_max": 650000,
        "bedrooms_min": 2,
        "bedrooms_max": 2,
    },
    "inversionista_renta_corta": {
        "descripcion": "Inversionista de renta corta, busca zona reconocida y buena conectividad, unidad pequeña.",
        "corregimientos": ["San Francisco", "Bella Vista", "El Cangrejo", "Marbella", "Obarrio"],
        "price_usd_min": 150000,
        "bedrooms_min": 1,
        "bedrooms_max": 2,
        "area_m2_max": 100,
    },
    "retirado_tranquilidad": {
        "descripcion": "Persona retirada que prioriza seguridad y amenidades, no depende de transporte diario.",
        "corregimientos": ["Betania"],  # mayor (seguridad + amenidades) combinado, ver sección 1
        "bedrooms_min": 2,
        "bedrooms_max": 3,
    },
    "presupuesto_ajustado_sin_auto": {
        "descripcion": "Presupuesto ajustado, depende de transporte público, unidad pequeña.",
        "corregimientos": ["Bella Vista", "El Cangrejo", "Marbella", "Obarrio", "Betania", "San Francisco"],
        "price_usd_max": 200000,
    },
}
print(f"{len(PERFILES_LIFESTYLE)} perfiles definidos.")


6 perfiles definidos.


In [5]:
def aplicar_filtro_estructural(df, perfil):
    m = df["corregimiento"].isin(perfil["corregimientos"])
    if "price_usd_min" in perfil:
        m &= df["price_usd"] >= perfil["price_usd_min"]
    if "price_usd_max" in perfil:
        m &= df["price_usd"] <= perfil["price_usd_max"]
    if "bedrooms_min" in perfil:
        m &= df["bedrooms"] >= perfil["bedrooms_min"]
    if "bedrooms_max" in perfil:
        m &= df["bedrooms"] <= perfil["bedrooms_max"]
    if "area_m2_min" in perfil:
        m &= df["area_m2"] >= perfil["area_m2_min"]
    if "area_m2_max" in perfil:
        m &= df["area_m2"] <= perfil["area_m2_max"]
    return m

ground_truth_estructural = {}
for nombre, perfil in PERFILES_LIFESTYLE.items():
    m = aplicar_filtro_estructural(df_catalogo, perfil)
    ground_truth_estructural[nombre] = set(df_catalogo.loc[m, "listing_id"])

pd.DataFrame([
    {"perfil": n, "n_matches_solo_estructural": len(ids)} for n, ids in ground_truth_estructural.items()
])


,perfil,n_matches_solo_estructural
0,familia_con_ninos,109
1,profesional_joven,236
2,pareja_presupuesto_medio,168
3,inversionista_renta_corta,257
4,retirado_tranquilidad,62
5,presupuesto_ajustado_sin_auto,169


## 3. Criterio cualitativo por perfil, sobre `descripcion`

Reconstruimos el criterio cualitativo usando `descripcion` (Notebook 6.2.1, sección 4.4 —
enriquecimiento de `pipeline/scraper/enriquecer_detalle.py`) en vez de `title`. `descripcion` es un
párrafo de texto libre real (~280-300 palabras cuando está completo), con vocabulario que sí
aparece de forma literal en el catálogo — verificado contra frecuencias reales antes de elegir cada
palabra clave, no adivinado. Mantenemos las mismas categorías conceptuales del intento anterior
(proximidad, ambiente/audiencia, amenidades), ajustadas al vocabulario que realmente aparece:

| Perfil | Palabras clave (frecuencia real en el catálogo) |
|---|---|
| familia_con_ninos | "familiar" (220), "colegio" (121), "escuela" (117) |
| profesional_joven | "restaurante" (591), "bar" (420), "vida nocturna" (34), "caminable" (13) — excluye "renta corta"/"airbnb" |
| pareja_presupuesto_medio | "tranquil-" (290), "céntric-" (125) — excluye "vida nocturna" |
| inversionista_renta_corta | "renta corta" (37), "airbnb" (30), "piscina" (766), "gimnasio" (681), "amoblado" (79) |
| retirado_tranquilidad | "tranquil-" (290), "residencial" (597) — excluye "vía principal"/"avenida" |
| presupuesto_ajustado_sin_auto | "metro" (277), "parada" (11), "transporte" (56) |

**Diferenciación deliberada `profesional_joven` vs. `inversionista_renta_corta`:** ambos comparten
vocabulario de "estilo de vida urbano" (restaurantes, vida nocturna) porque el propio texto
publicitario del mercado los mezcla — la muestra de PH The Hub decía literalmente *"diseñado para
profesionales y viajeros... una sólida inversión"* en el mismo párrafo. En vez de competir por las
mismas palabras, usamos `"renta corta"` / `"airbnb"` como señal exclusiva de intención de
inversión (no aparecen en el vocabulario de `profesional_joven`) y las excluimos explícitamente del
otro perfil.


**Regla explícita para `descripcion_fuente == "ninguna"` (9 filas) y `"preview"` (31 filas) —
no quedan ambiguos:**

- **`"ninguna"` (sin texto de ningún tipo):** excluidas de **todo** ground truth cualitativo, sin
  importar si pasan el filtro estructural. No hay evidencia textual a favor ni en contra — ausencia
  de evidencia no es evidencia de match, así que no las contamos como positivas.
- **`"preview"` (texto corto real, pero mucho más corto que una descripción completa):** se evalúan
  con el mismo filtro que las de `"completa"` — si el texto corto contiene la palabra clave, cuentan
  como match. No las tratamos distinto en la lógica del filtro, pero reportamos por separado cuántos
  matches de cada perfil vienen de `"preview"` vs. `"completa"`, porque un match sobre un texto de
  una línea es evidencia más débil que uno sobre un párrafo completo.


In [6]:
KEYWORDS_CUALITATIVOS = {
    "familia_con_ninos": {"incluye": ["familiar", "colegio", "escuela"]},
    "profesional_joven": {"incluye": ["restaurante", "bar", "vida nocturna", "caminable"],
                           "excluye": ["renta corta", "airbnb"]},
    "pareja_presupuesto_medio": {"incluye": ["tranquil", "céntric"], "excluye": ["vida nocturna"]},
    "inversionista_renta_corta": {"incluye": ["renta corta", "airbnb", "piscina", "gimnasio", "amoblado"]},
    "retirado_tranquilidad": {"incluye": ["tranquil", "residencial"], "excluye": ["vía principal", "avenida"]},
    "presupuesto_ajustado_sin_auto": {"incluye": ["metro", "parada", "transporte"]},
}

def aplicar_filtro_cualitativo(df, criterio):
    descripciones = df["descripcion"].fillna("").str.lower()
    tiene_texto = df["descripcion_fuente"] != "ninguna"  # regla explicita: sin texto, no cuenta como match
    incluye = criterio.get("incluye", [])
    m = descripciones.str.contains("|".join(incluye), regex=True, na=False) if incluye else pd.Series(True, index=df.index)
    for kw in criterio.get("excluye", []):
        m &= ~descripciones.str.contains(kw, regex=False, na=False)
    return m & tiene_texto

ground_truth_final = {}
filas_resumen = []
for nombre, perfil in PERFILES_LIFESTYLE.items():
    m_estructural = aplicar_filtro_estructural(df_catalogo, perfil)
    m_cualitativo = aplicar_filtro_cualitativo(df_catalogo, KEYWORDS_CUALITATIVOS[nombre])
    m_final = m_estructural & m_cualitativo
    ids_final = set(df_catalogo.loc[m_final, "listing_id"])
    ground_truth_final[nombre] = ids_final
    fuente_counts = df_catalogo.loc[m_final, "descripcion_fuente"].value_counts().to_dict()
    filas_resumen.append({
        "perfil": nombre,
        "n_solo_estructural": int(m_estructural.sum()),
        "n_estructural_y_cualitativo": len(ids_final),
        "de_los_cuales_completa": fuente_counts.get("completa", 0),
        "de_los_cuales_preview": fuente_counts.get("preview", 0),
    })

df_resumen_final = pd.DataFrame(filas_resumen)
df_resumen_final


,perfil,n_solo_estructural,n_estructural_y_cualitativo,de_los_cuales_completa,de_los_cuales_preview
0,familia_con_ninos,109,49,49,0
1,profesional_joven,236,153,152,1
2,pareja_presupuesto_medio,168,60,60,0
3,inversionista_renta_corta,257,199,197,2
4,retirado_tranquilidad,62,43,43,0
5,presupuesto_ajustado_sin_auto,169,73,72,1


In [7]:
for nombre, ids in ground_truth_final.items():
    assert len(ids) > 0, f"Perfil '{nombre}' quedó sin ningún match tras el filtro cualitativo sobre descripcion"
print("Los 6 perfiles tienen al menos un match en el ground truth reconstruido sobre descripcion.")


Los 6 perfiles tienen al menos un match en el ground truth reconstruido sobre descripcion.


**Resultado real:** los 6 perfiles quedan con ground truth no vacío usando `descripcion` —
contraste directo con el intento anterior sobre `title`, que colapsaba 3 de 6 perfiles a 0. El
rango va de 43 (`retirado_tranquilidad`) a 199 (`inversionista_renta_corta`) matches. La enorme
mayoría de los matches provienen de `descripcion_fuente == "completa"` — los matches sobre
`"preview"` son una fracción mínima en cada perfil (ver columna `de_los_cuales_preview` arriba),
consistente con que el texto de preview es demasiado corto para contener la mayoría de las frases
clave.


## 4. Análisis de overlap entre perfiles

Calculamos el overlap tanto sobre el filtro solo-estructural (`ground_truth_estructural`, para
comparar contra el intento anterior) como sobre el ground truth final (estructural + cualitativo
sobre `descripcion`), para ver cuánto reduce el criterio cualitativo el overlap real.


In [8]:
def calcular_overlap(diccionario_sets):
    nombres = list(diccionario_sets.keys())
    filas = []
    for i, a in enumerate(nombres):
        for b in nombres[i + 1:]:
            set_a, set_b = diccionario_sets[a], diccionario_sets[b]
            interseccion = set_a & set_b
            union = set_a | set_b
            jaccard = len(interseccion) / len(union) if union else 0.0
            pct_del_menor = len(interseccion) / min(len(set_a), len(set_b)) * 100 if min(len(set_a), len(set_b)) > 0 else 0.0
            filas.append({"perfil_a": a, "perfil_b": b, "n_interseccion": len(interseccion),
                          "jaccard_pct": round(jaccard * 100, 1), "pct_del_menor": round(pct_del_menor, 1)})
    return pd.DataFrame(filas).sort_values("pct_del_menor", ascending=False)

df_overlap_estructural = calcular_overlap(ground_truth_estructural)
df_overlap_final = calcular_overlap(ground_truth_final)

print("=== Overlap SOLO estructural (antes del criterio cualitativo) ===")
display(df_overlap_estructural.head(8))
print("\n=== Overlap FINAL (estructural + cualitativo sobre descripcion) ===")
display(df_overlap_final.head(8))


=== Overlap SOLO estructural (antes del criterio cualitativo) ===


,perfil_a,perfil_b,n_interseccion,jaccard_pct,pct_del_menor
6,profesional_joven,inversionista_renta_corta,162,48.9,68.6
8,profesional_joven,presupuesto_ajustado_sin_auto,93,29.8,55.0
13,inversionista_renta_corta,presupuesto_ajustado_sin_auto,76,21.7,45.0
9,pareja_presupuesto_medio,inversionista_renta_corta,72,20.4,42.9
14,retirado_tranquilidad,presupuesto_ajustado_sin_auto,25,12.1,40.3
5,profesional_joven,pareja_presupuesto_medio,67,19.9,39.9
3,familia_con_ninos,retirado_tranquilidad,14,8.9,22.6
10,pareja_presupuesto_medio,retirado_tranquilidad,2,0.9,3.2



=== Overlap FINAL (estructural + cualitativo sobre descripcion) ===


,perfil_a,perfil_b,n_interseccion,jaccard_pct,pct_del_menor
6,profesional_joven,inversionista_renta_corta,98,38.6,64.1
8,profesional_joven,presupuesto_ajustado_sin_auto,36,18.9,49.3
13,inversionista_renta_corta,presupuesto_ajustado_sin_auto,35,14.8,47.9
9,pareja_presupuesto_medio,inversionista_renta_corta,28,12.1,46.7
5,profesional_joven,pareja_presupuesto_medio,27,14.5,45.0
14,retirado_tranquilidad,presupuesto_ajustado_sin_auto,10,9.4,23.3
3,familia_con_ninos,retirado_tranquilidad,8,9.5,18.6
0,familia_con_ninos,profesional_joven,0,0.0,0.0


In [9]:
fila_pj_irc_antes = df_overlap_estructural[
    ((df_overlap_estructural["perfil_a"] == "profesional_joven") & (df_overlap_estructural["perfil_b"] == "inversionista_renta_corta")) |
    ((df_overlap_estructural["perfil_a"] == "inversionista_renta_corta") & (df_overlap_estructural["perfil_b"] == "profesional_joven"))
]
fila_pj_irc_despues = df_overlap_final[
    ((df_overlap_final["perfil_a"] == "profesional_joven") & (df_overlap_final["perfil_b"] == "inversionista_renta_corta")) |
    ((df_overlap_final["perfil_a"] == "inversionista_renta_corta") & (df_overlap_final["perfil_b"] == "profesional_joven"))
]
print("profesional_joven vs inversionista_renta_corta -- SOLO estructural:")
print(fila_pj_irc_antes.to_string(index=False))
print("\nprofesional_joven vs inversionista_renta_corta -- FINAL (con criterio cualitativo):")
print(fila_pj_irc_despues.to_string(index=False))


profesional_joven vs inversionista_renta_corta -- SOLO estructural:
         perfil_a                  perfil_b  n_interseccion  jaccard_pct  pct_del_menor
profesional_joven inversionista_renta_corta             162         48.9           68.6

profesional_joven vs inversionista_renta_corta -- FINAL (con criterio cualitativo):
         perfil_a                  perfil_b  n_interseccion  jaccard_pct  pct_del_menor
profesional_joven inversionista_renta_corta              98         38.6           64.1


**Caso específico — `profesional_joven` vs. `inversionista_renta_corta`:** el overlap baja con
el criterio cualitativo (ver celda anterior para las cifras exactas), pero **no desaparece**. Ambos
perfiles siguen compartiendo el mismo grupo base de corregimientos (Bella Vista, El Cangrejo,
Marbella, Obarrio), y aunque diferenciamos por vocabulario de intención (`"renta corta"`/`"airbnb"`
exclusivo de `inversionista_renta_corta`), ambos perfiles todavía capturan listings cuya descripción
menciona restaurantes/vida urbana sin mencionar explícitamente intención de inversión — esos caen
en `profesional_joven` pero también cumplen los rangos estructurales de `inversionista_renta_corta`.
La mejora es real (el overlap baja), pero la distinción entre "vive aquí" y "invierte aquí" en el
vocabulario real del mercado panameño de bienes raíces es más difusa de lo que el diseño original de
6 perfiles separados asumía — esto queda documentado como limitación del conjunto de referencia, no
oculto.


**Dos hipótesis abiertas para el 64.1% de solapamiento restante — no elegimos una sobre la
otra:**

1. **Limitación del método de keywords.** Nuestro criterio cualitativo depende de un vocabulario
   fijo, elegido por nosotros, sobre un campo de texto (`descripcion`) que mezcla lenguaje de
   "estilo de vida" e "inversión" en el mismo párrafo (ver la muestra de PH The Hub, sección 3). Es
   posible que un método más fino (embeddings semánticos en vez de keywords literales, o un
   vocabulario más específico) separe mejor ambos perfiles sobre exactamente el mismo catálogo.
2. **Solapamiento genuino de mercado.** Es igualmente posible que el tipo de propiedad que atrae a
   un profesional joven que quiere vivir ahí sea, objetivamente, muy parecido o el mismo tipo de
   propiedad que atrae a un inversionista de renta corta cuyo huésped objetivo *es* ese mismo
   profesional joven — un PH pequeño, bien ubicado, cerca de vida urbana, en Bella Vista/El
   Cangrejo/Marbella/Obarrio, sirve igual de bien a ambos casos de uso porque el mercado los
   diseñó para lo mismo.

No tenemos evidencia en este proyecto para descartar ninguna de las dos — ambas quedan como
hipótesis abiertas hasta que haya una señal independiente (datos de uso real, o una revisión
cualitativa manual de una muestra) que las distinga.


## 5. Advertencia de validación circular (obligatoria)

El ground truth de cada perfil combina atributos estructurados (`corregimiento`, `price_usd`,
`bedrooms`, `area_m2`) — que el componente estructurado del scorer híbrido (peso 0.6) usa
directamente — con un criterio cualitativo sobre `descripcion`, un campo independiente de esos
atributos. Esto reduce la circularidad respecto al primer intento (que era 100% estructural), pero
**no la elimina**: el componente estructurado sigue determinando qué zona/precio/tamaño calza,
y el criterio cualitativo actúa como filtro adicional sobre ese mismo subconjunto, no como una
fuente de verdad independiente. Precision@k medido contra este conjunto sigue midiendo, en buena
parte, qué tan bien el modelo reproduce reglas que nosotros mismos escribimos — no mide qué tan
bien el matching completo se alinea con preferencias reales de usuarios, porque no existen datos de
comportamiento real de usuarios en este proyecto.

Esta limitación es estructural, no un error de diseño de este conjunto: no hay una alternativa sin
datos de comportamiento real. El valor de este conjunto es servir de humo de prueba (`smoke test`)
del pipeline de matching de principio a fin, y de guía para iterar el componente semántico (¿el
ranking dentro de cada grupo de match estructurado+cualitativo tiene sentido semánticamente, más
allá de las palabras clave literales que usamos para construirlo?) — no de validación de producto.


## 6. Embeddings sobre el catálogo completo y los 6 perfiles

Generamos embeddings con `gemini-embedding-001` (3072 dimensiones, confirmado en la sección de
verificación de entorno) para: (a) cada propiedad del catálogo filtrado (`df_catalogo`, n=1,110),
usando `descripcion` cuando existe texto (`completa` o `preview`) y el `title` como respaldo para
las 9 filas `descripcion_fuente == "ninguna"` — no dejamos ninguna propiedad sin vector; y (b) el
texto de lifestyle de cada uno de los 6 perfiles.

**Historial real de la generación, no idealizado:** el primer intento de correr esto dentro de este
notebook agotó una cuota de la API a los ~17 minutos (`429 RESOURCE_EXHAUSTED`, plan gratuito) sin
guardar ningún progreso parcial — perdimos toda la corrida. Diagnosticamos el problema con pruebas
en escalera (10 → lotes de 20) fuera del notebook antes de reintentar aquí, confirmamos que el SDK
ya usa `batchEmbedContents` (una sola request HTTP por lote, no una por fila), y finalmente
activamos facturación de pago (crédito prepago, sin recarga automática) para dejar de operar contra
el límite del plan gratuito. Con eso, generamos los 1,110 + 6 embeddings en una corrida separada
(script con circuit breaker: ante un 429, espera 90s una sola vez, reintenta una sola vez, y si
vuelve a fallar se detiene por completo sin más reintentos) con persistencia incremental a disco
después de cada lote — así que un fallo a mitad de camino no habría perdido el progreso ya hecho,
a diferencia del primer intento. Esa corrida terminó sin ningún 429. Cargamos su resultado aquí en
vez de volver a llamar la API desde este notebook, para no arriesgar otro consumo de cuota
innecesario regenerando lo que ya existe.


In [10]:
import pickle

RUTA_EMBEDDINGS_CATALOGO = REPO_ROOT / "pipeline" / "models" / "embeddings_catalogo_6_2_3_raw.pkl"
RUTA_EMBEDDINGS_PERFILES = REPO_ROOT / "pipeline" / "models" / "embeddings_perfiles_6_2_3_raw.pkl"

with open(RUTA_EMBEDDINGS_CATALOGO, "rb") as f:
    _catalogo_guardado = pickle.load(f)
with open(RUTA_EMBEDDINGS_PERFILES, "rb") as f:
    _perfiles_guardado = pickle.load(f)

listing_ids_catalogo = _catalogo_guardado["id"]
matriz_catalogo = np.vstack(_catalogo_guardado["vector"])
embeddings_perfiles = dict(zip(_perfiles_guardado["id"], _perfiles_guardado["vector"]))

# Verificacion de integridad: mismo listing_id, mismo orden que df_catalogo -- si esto
# fallara, un merge por posicion en vez de por listing_id seria silenciosamente incorrecto.
assert listing_ids_catalogo == df_catalogo["listing_id"].tolist(), \
    "El orden de listing_id en el archivo de embeddings no coincide con df_catalogo"
assert set(embeddings_perfiles.keys()) == set(PERFILES_LIFESTYLE.keys()), \
    "Los perfiles del archivo de embeddings no coinciden con PERFILES_LIFESTYLE"
assert matriz_catalogo.shape == (len(df_catalogo), 3072)

print(f"Embeddings cargados: {matriz_catalogo.shape[0]} propiedades, "
      f"{len(embeddings_perfiles)} perfiles, dimensión {matriz_catalogo.shape[1]}.")


Embeddings cargados: 1110 propiedades, 6 perfiles, dimensión 3072.


### 6.1 Persistencia de los embeddings

Ya están persistidos en `pipeline/models/` (`embeddings_catalogo_6_2_3_raw.pkl`,
`embeddings_perfiles_6_2_3_raw.pkl`) desde la corrida que los generó — no se regeneran cada vez que
se re-ejecuta este notebook.


## 7. Scorer híbrido (estructurado 0.6 / semántico 0.4)

Definimos, para cada perfil, un score por propiedad sobre **todo** `df_catalogo` (no solo el
subconjunto de ground truth):

- **Componente estructurado (peso 0.6):** promedio de indicadores binarios sobre los mismos
  criterios que definen el ground truth (corregimiento en la lista del perfil, precio en rango,
  habitaciones en rango, área en rango) — a diferencia del ground truth (que exige los 4 a la vez,
  AND), aquí promediamos para dar crédito parcial a una propiedad que cumple algunos criterios pero
  no todos, y así poder rankear el catálogo completo, no solo el subconjunto que ya cumple todo.
- **Componente semántico (peso 0.4):** similitud coseno entre el embedding de la propiedad y el
  embedding del texto de lifestyle del perfil, normalizada min-max sobre las 1,110 propiedades para
  ese perfil (la similitud coseno cruda ocupa un rango angosto, normalizarla evita que quede
  aplastada frente al componente estructurado que sí cubre [0,1] completo).


In [11]:
def similitud_coseno_matriz(matriz, vector):
    norm_matriz = matriz / np.linalg.norm(matriz, axis=1, keepdims=True)
    norm_vector = vector / np.linalg.norm(vector)
    return norm_matriz @ norm_vector

def score_estructurado(df, perfil):
    componentes = [df["corregimiento"].isin(perfil["corregimientos"]).astype(float)]
    if "price_usd_min" in perfil:
        componentes.append((df["price_usd"] >= perfil["price_usd_min"]).astype(float))
    if "price_usd_max" in perfil:
        componentes.append((df["price_usd"] <= perfil["price_usd_max"]).astype(float))
    if "bedrooms_min" in perfil:
        componentes.append((df["bedrooms"] >= perfil["bedrooms_min"]).astype(float))
    if "bedrooms_max" in perfil:
        componentes.append((df["bedrooms"] <= perfil["bedrooms_max"]).astype(float))
    if "area_m2_min" in perfil:
        componentes.append((df["area_m2"] >= perfil["area_m2_min"]).astype(float))
    if "area_m2_max" in perfil:
        componentes.append((df["area_m2"] <= perfil["area_m2_max"]).astype(float))
    return pd.concat(componentes, axis=1).mean(axis=1)

PESO_ESTRUCTURADO = 0.6
PESO_SEMANTICO = 0.4

scores_hibridos = {}
for nombre, perfil in PERFILES_LIFESTYLE.items():
    s_estructurado = score_estructurado(df_catalogo, perfil).to_numpy()

    s_semantico_crudo = similitud_coseno_matriz(matriz_catalogo, embeddings_perfiles[nombre])
    minimo, maximo = s_semantico_crudo.min(), s_semantico_crudo.max()
    s_semantico = (s_semantico_crudo - minimo) / (maximo - minimo) if maximo > minimo else np.zeros_like(s_semantico_crudo)

    hibrido = PESO_ESTRUCTURADO * s_estructurado + PESO_SEMANTICO * s_semantico
    scores_hibridos[nombre] = pd.DataFrame({
        "listing_id": listing_ids_catalogo,
        "score_estructurado": s_estructurado,
        "score_semantico": s_semantico,
        "score_hibrido": hibrido,
    }).sort_values("score_hibrido", ascending=False).reset_index(drop=True)

print("Scores híbridos calculados para los 6 perfiles.")


Scores híbridos calculados para los 6 perfiles.


## 8. Precision@k (@3, @5, @10)

Calculamos Precision@k tomando el top-k de `score_hibrido` por perfil y comparándolo contra
`ground_truth_final` (sección 3) — el conjunto que combina filtro estructural + criterio
cualitativo sobre `descripcion`.


In [12]:
def precision_en_k(ranking_listing_ids, ground_truth_ids, k):
    top_k = ranking_listing_ids[:k]
    aciertos = sum(1 for lid in top_k if lid in ground_truth_ids)
    return aciertos / k

filas_precision = []
for nombre in PERFILES_LIFESTYLE:
    ranking = scores_hibridos[nombre]["listing_id"].tolist()
    gt = ground_truth_final[nombre]
    filas_precision.append({
        "perfil": nombre,
        "n_ground_truth": len(gt),
        "precision_at_3": round(precision_en_k(ranking, gt, 3), 3),
        "precision_at_5": round(precision_en_k(ranking, gt, 5), 3),
        "precision_at_10": round(precision_en_k(ranking, gt, 10), 3),
    })

df_precision = pd.DataFrame(filas_precision)
df_precision


,perfil,n_ground_truth,precision_at_3,precision_at_5,precision_at_10
0,familia_con_ninos,49,0.667,0.8,0.7
1,profesional_joven,153,0.000,0.0,0.0
2,pareja_presupuesto_medio,60,0.000,0.2,0.4
3,inversionista_renta_corta,199,1.000,1.0,1.0
4,retirado_tranquilidad,43,1.000,0.8,0.7
5,presupuesto_ajustado_sin_auto,73,0.000,0.4,0.6


### 8.1 Baseline solo-estructurado (1.0 / 0.0) — mismo embeddings, sin volver a llamar la API

Recalculamos Precision@k con el peso del scorer en 1.0 estructurado / 0.0 semántico, reusando
la misma matriz de embeddings ya cargada — sin ninguna llamada nueva a la API. El objetivo es ver,
perfil por perfil, si el componente semántico (0.4) aporta lift real sobre el filtro estructurado
solo, o si en algún caso lo empeora.


In [13]:
filas_comparativas = []
for nombre, perfil in PERFILES_LIFESTYLE.items():
    s_estructurado = score_estructurado(df_catalogo, perfil).to_numpy()
    ranking_solo_estructurado = [
        listing_ids_catalogo[i] for i in np.argsort(-s_estructurado, kind="stable")
    ]
    ranking_hibrido = scores_hibridos[nombre]["listing_id"].tolist()
    gt = ground_truth_final[nombre]

    fila = {"perfil": nombre}
    for k in (3, 5, 10):
        fila[f"P{k}_solo_estructurado"] = round(precision_en_k(ranking_solo_estructurado, gt, k), 3)
        fila[f"P{k}_hibrido"] = round(precision_en_k(ranking_hibrido, gt, k), 3)
        fila[f"lift_P{k}"] = round(fila[f"P{k}_hibrido"] - fila[f"P{k}_solo_estructurado"], 3)
    filas_comparativas.append(fila)

df_comparativa = pd.DataFrame(filas_comparativas)
df_comparativa


,perfil,P3_solo_estructurado,P3_hibrido,lift_P3,P5_solo_estructurado,P5_hibrido,lift_P5,P10_solo_estructurado,P10_hibrido,lift_P10
0,familia_con_ninos,0.667,0.667,0.000,0.8,0.8,0.0,0.9,0.7,-0.2
1,profesional_joven,0.667,0.000,-0.667,0.8,0.0,-0.8,0.9,0.0,-0.9
2,pareja_presupuesto_medio,0.333,0.000,-0.333,0.4,0.2,-0.2,0.3,0.4,0.1
3,inversionista_renta_corta,0.667,1.000,0.333,0.8,1.0,0.2,0.9,1.0,0.1
4,retirado_tranquilidad,0.000,1.000,1.000,0.4,0.8,0.4,0.7,0.7,0.0
5,presupuesto_ajustado_sin_auto,0.333,0.000,-0.333,0.4,0.4,0.0,0.2,0.6,0.4


**Lectura perfil por perfil, no agregada — el lift semántico va en las dos direcciones:**

- **`inversionista_renta_corta`:** lift positivo y consistente (0.667→1.0, 0.8→1.0, 0.9→1.0). El
  componente semántico ayuda de forma clara y across-the-board.
- **`retirado_tranquilidad`:** el lift más dramático de los 6 — P@3 pasa de 0.0 (solo estructurado)
  a 1.0 (híbrido). El componente semántico aporta señal que el filtro estructurado solo no captura
  en absoluto en el top-3.
- **`profesional_joven`:** lift **negativo y severo** — P@3/P@5/P@10 caen de 0.667/0.8/0.9 (solo
  estructurado, un baseline razonablemente bueno) a 0.0/0.0/0.0 (híbrido). **Esto contradice la
  hipótesis que habíamos planteado en la primera versión de la sección 9** (que el problema era
  solo la definición de ground truth, y que el baseline estructurado también daría 0). No da 0 —
  da un baseline sólido. El componente semántico es el que activamente empeora este perfil
  específico. Desarrollamos esto en la sección 9.
- **`pareja_presupuesto_medio`** y **`presupuesto_ajustado_sin_auto`:** patrón mixto, no uniforme —
  el semántico empeora P@3 en ambos, pero mejora P@10 en `presupuesto_ajustado_sin_auto` (0.2→0.6)
  y lo empeora en `pareja_presupuesto_medio` (0.3→0.4, leve mejora en realidad). Ver sección 9 para
  la separación explícita de este patrón frente al de `profesional_joven`.
- **`familia_con_ninos`:** el más estable de los 6 — P@3/P@5 casi idénticos entre ambos scorers,
  P@10 baja levemente con el híbrido (0.9→0.7).

Recordamos también la advertencia de validación circular de la sección 5: estos números miden,
en buena parte, qué tan bien cada versión del scorer reproduce reglas que nosotros mismos
escribimos para construir el ground truth — no una validación contra preferencias reales de
usuarios.


### 8.2 Síntesis — el lift promedio no cuenta la historia real

No dejamos que la tabla hable sola. Calculamos el lift promedio (híbrido − solo-estructurado) sobre
los 6 perfiles, para cada k.


In [14]:
lift_promedio = df_comparativa[["lift_P3", "lift_P5", "lift_P10"]].mean().round(3)
print(lift_promedio)


lift_P3    -0.000
lift_P5    -0.067
lift_P10   -0.083
dtype: float64


**El promedio (~0.00 / -0.07 / -0.08) sugiere, a primera vista, que el componente semántico es
prácticamente neutro o levemente perjudicial en promedio.** Esa lectura agregada es engañosa y la
rechazamos explícitamente: el efecto real es **condicional y heterogéneo por perfil**, no una
mejora (ni un daño) universal y parejo. Tres perfiles concentran toda la señal:

- **`retirado_tranquilidad`** (+1.0 en P@3) e **`inversionista_renta_corta`** (+0.333 en P@3, positivo
  en las 3 métricas): el componente semántico aporta lift real y claro. Aquí el peso 0.4 se justifica.
- **`profesional_joven`** (-0.667 a -0.9 según k): el componente semántico interactúa negativamente
  con la regla de exclusión de la sección 3 (documentado en la sección 9.2) — no es que el semántico
  sea malo en general, es que en este perfil específico su noción de similitud coincide con
  contenido que nuestra propia regla de ground truth excluye por diseño.
- Los otros 3 perfiles (`familia_con_ninos`, `pareja_presupuesto_medio`,
  `presupuesto_ajustado_sin_auto`) quedan cerca de neutro o con patrones mixtos por k (sección 9.1).

El promedio de -0.07/-0.08 es aritméticamente correcto pero **oculta que el componente semántico no
tiene un efecto único** — tiene un efecto fuertemente positivo en 2 perfiles y fuertemente negativo
en 1, con el resto cerca de neutro. Reportar solo el promedio agregado habría escondido tanto el
mejor caso de uso del componente semántico como su modo de falla más claro.


## 9. Dos fallas de naturaleza distinta — no la misma nota interpretativa

Los 6 perfiles no fallan de la misma manera. Separamos explícitamente dos patrones distintos en
vez de agruparlos bajo una sola explicación genérica.

### 9.1 Tipo A — el modelo encuentra lo relevante pero no lo prioriza arriba

`pareja_presupuesto_medio` y `presupuesto_ajustado_sin_auto`: P@3 bajo (0.0 ambos) pero la señal
no desaparece al ampliar k — `presupuesto_ajustado_sin_auto` sube a P@10=0.6 (mejor que su propio
baseline solo-estructurado, 0.2), y `pareja_presupuesto_medio` sube a P@10=0.4. El patrón típico de
"el contenido relevante existe en el catálogo y el componente semántico lo encuentra, pero no
siempre lo empuja a las primeras 3 posiciones" — no es un fallo estructural del scorer, es una
cuestión de qué tan afinado está el ranking en las posiciones más altas. Documentamos con cautela:
el lift de `presupuesto_ajustado_sin_auto` en P@10 (0.2→0.6) es una mejora real del componente
semántico, no ausencia de mejora — este perfil no es un caso limpio de "sin lift", es un caso de
"lift que aparece más tarde en el ranking".


### 9.2 Tipo B — el modelo rankea alto exactamente lo que el ground truth excluye por construcción

`profesional_joven` es el único caso de los 6 con 0.000 en las 3 métricas simultáneamente, **y con
un baseline solo-estructurado que NO es cero** (0.667 / 0.8 / 0.9 — sección 8.1). Esto es
información decisiva que no teníamos en la primera versión de este notebook: el filtro estructurado
solo ya encuentra buenos matches. **Es específicamente el componente semántico el que degrada este
perfil de un baseline sólido a cero.** Investigamos por qué, inspeccionando el top-10 real del
ranking híbrido.


In [15]:
ranking_profesional = scores_hibridos["profesional_joven"].head(10).merge(
    df_catalogo[["listing_id", "descripcion", "corregimiento"]], on="listing_id"
)

KW_INCLUYE = KEYWORDS_CUALITATIVOS["profesional_joven"]["incluye"]
KW_EXCLUYE = KEYWORDS_CUALITATIVOS["profesional_joven"]["excluye"]

for _, fila in ranking_profesional.iterrows():
    texto = str(fila["descripcion"]).lower()
    incluye_hit = [k for k in KW_INCLUYE if k in texto]
    excluye_hit = [k for k in KW_EXCLUYE if k in texto]
    en_ground_truth = fila["listing_id"] in ground_truth_final["profesional_joven"]
    print(f"id={fila['listing_id']} score_estructurado={fila['score_estructurado']:.2f} "
          f"incluye={incluye_hit} excluye={excluye_hit} en_ground_truth={en_ground_truth}")


id=140041 score_estructurado=1.00 incluye=[] excluye=['renta corta'] en_ground_truth=False
id=140040 score_estructurado=1.00 incluye=[] excluye=[] en_ground_truth=False
id=141249 score_estructurado=1.00 incluye=[] excluye=['renta corta'] en_ground_truth=False
id=140043 score_estructurado=1.00 incluye=['bar'] excluye=['renta corta'] en_ground_truth=False
id=140042 score_estructurado=1.00 incluye=['bar'] excluye=['renta corta'] en_ground_truth=False
id=140081 score_estructurado=1.00 incluye=['restaurante'] excluye=['renta corta', 'airbnb'] en_ground_truth=False
id=141250 score_estructurado=1.00 incluye=['bar'] excluye=['renta corta'] en_ground_truth=False
id=141251 score_estructurado=1.00 incluye=['bar'] excluye=['renta corta'] en_ground_truth=False
id=140229 score_estructurado=1.00 incluye=[] excluye=[] en_ground_truth=False
id=140227 score_estructurado=1.00 incluye=[] excluye=[] en_ground_truth=False


**Causa raíz:** el top-10 por score híbrido son, casi todos, propiedades con
`score_estructurado == 1.0` cuyo texto de `descripcion` contiene palabras clave relevantes al
perfil ("bar", "restaurante") **pero también "renta corta"** — la palabra que agregamos
explícitamente como criterio de *exclusión* de `profesional_joven` (sección 3) para diferenciarlo
de `inversionista_renta_corta`. El catálogo real tiene listings que se anuncian simultáneamente
como "unidad moderna y céntrica con Sky Bar" *y* "diseñado para inversionistas de renta corta" **en
el mismo texto**. Nuestra regla de exclusión saca a estas propiedades del ground truth; el
componente semántico (que no sabe nada de esa regla) las sigue rankeando arriba porque su texto es,
en general, más rico en lenguaje de "ubicación céntrica, moderno, urbano" que el de los listings
que sí quedan dentro del ground truth — probablemente porque el copy de marketing dirigido a
inversionistas tiende a ser más elaborado que el de un listing residencial simple, lo que lo acerca
más, en el espacio de embeddings, a cualquier descripción de lifestyle urbano.

**Verificación contra la hipótesis inicial:** antes de tener el baseline solo-estructurado
considerábamos posible que el 0.0 fuera puramente un artefacto de cómo definimos el ground truth.
El baseline **no** da 0.0 — da 0.9 en P@10 (sección 8.1). Eso descarta esa lectura. El mecanismo es
más específico: el componente semántico activamente prefiere el copy de marketing de inversión
sobre el copy residencial simple, y nuestra regla de exclusión (necesaria para diferenciar de
`inversionista_renta_corta`, sección 4) es exactamente lo que convierte esa preferencia semántica en
un 0.0 en vez de un número intermedio.

No tocamos la regla de exclusión — se queda como está, documentada aquí como hallazgo metodológico
válido, no como algo a corregir en este notebook.

Contraste directo con `inversionista_renta_corta`, cuyo Precision@10=1.0 sí verificamos como
genuino: sus 10 primeros listings contienen literalmente `"renta corta"`, `"airbnb"`, `"piscina"` o
`"gimnasio"` en el texto completo (no solo en los primeros caracteres) — no es un artefacto de
conteo, y ahí el componente semántico sí está alineado con el ground truth.

### 9.3 Lectura precisa del hallazgo, tras el sensitivity sweep de la sección 10

El sweep de pesos (sección 10) confirma que esto no es un problema de magnitud del peso semántico
— ningún valor entre 0.2 y 0.4 cambia el resultado. Eso obliga a una lectura más precisa de la
causa raíz que la de la sección 9.2: **no es que el embedding "prefiera lenguaje de inversión" de
forma arbitraria — es que el embedding correlaciona con la riqueza del copy de marketing en
general, y las propiedades con copy más elaborado resultan ser, en este catálogo, sistemáticamente
las mismas que también se anuncian para renta corta.** No son dos fenómenos independientes que
coinciden por casualidad — es un solo patrón real del mercado: los desarrollos que invierten más en
redacción publicitaria (mencionar Sky Bar, coworking, gimnasio, vista, ubicación estratégica) son,
en este catálogo, los mismos que también apuntan a inversionistas de renta corta. Nuestra regla de
exclusión intenta separar artificialmente dos audiencias (residente joven vs. inversionista) que el
propio texto del anuncio no separa limpiamente — el mercado real las anuncia juntas en la misma
propiedad, con el mismo párrafo.

**Esto es una observación válida sobre el mercado inmobiliario panameño de este segmento, no
únicamente una falla del método de keywords.** Documentamos la limitación explícitamente:

- **Limitación conocida:** una palabra clave de exclusión (`"renta corta"`, `"airbnb"`) es una señal
  frágil para separar audiencias objetivo, porque el texto de marketing real mezcla ambas
  intenciones en el mismo párrafo con más frecuencia de lo que el diseño de perfiles separados
  asumía.
- **Dirección de trabajo futura, no implementada aquí:** la solución real no es ajustar el peso del
  scorer — el sweep ya muestra que eso no cambia nada — sino reemplazar la keyword de exclusión
  frágil por una etiqueta estructurada de audiencia objetivo (ej. `audiencia: residente | inversionista | ambas`)
  extraída con un criterio más robusto que coincidencia de substring. El Quality Scorer (6.2.6,
  basado en LLM) es un candidato natural para generar esa etiqueta en una iteración futura, dado que
  ya está diseñado para leer texto libre de listings — no lo implementamos en este notebook, queda
  anotado como el próximo paso lógico para esta limitación específica.


## 10. Sensitivity sweep de pesos (estructurado/semántico)

Recalculamos Precision@k para pesos alternativos, reusando los mismos embeddings ya cargados — sin
ninguna llamada nueva a la API. Objetivo: ver si algún peso reduce el daño en `profesional_joven`
sin sacrificar la ganancia en `retirado_tranquilidad`/`inversionista_renta_corta`. No cambiamos el
peso de producción (0.6/0.4) en este notebook — solo reportamos evidencia para decidir.


In [16]:
PESOS_SWEEP = [(1.0, 0.0), (0.8, 0.2), (0.7, 0.3), (0.6, 0.4), (0.5, 0.5)]

resultados_sweep = {}
for peso_estr, peso_sem in PESOS_SWEEP:
    filas = []
    for nombre, perfil in PERFILES_LIFESTYLE.items():
        s_estructurado = score_estructurado(df_catalogo, perfil).to_numpy()
        s_semantico_crudo = similitud_coseno_matriz(matriz_catalogo, embeddings_perfiles[nombre])
        mn, mx = s_semantico_crudo.min(), s_semantico_crudo.max()
        s_semantico = (s_semantico_crudo - mn) / (mx - mn)
        hibrido = peso_estr * s_estructurado + peso_sem * s_semantico
        ranking = [listing_ids_catalogo[i] for i in np.argsort(-hibrido, kind="stable")]
        gt = ground_truth_final[nombre]
        filas.append({
            "perfil": nombre,
            "P3": precision_en_k(ranking, gt, 3),
            "P5": precision_en_k(ranking, gt, 5),
            "P10": precision_en_k(ranking, gt, 10),
        })
    resultados_sweep[f"{peso_estr}/{peso_sem}"] = pd.DataFrame(filas).set_index("perfil")

for combo, df_r in resultados_sweep.items():
    print(f"=== estructurado={combo.split('/')[0]} / semántico={combo.split('/')[1]} ===")
    print(df_r)
    print("promedio:", df_r.mean().round(3).to_dict())
    print()


=== estructurado=1.0 / semántico=0.0 ===
                                     P3   P5  P10
perfil                                           
familia_con_ninos              0.666667  0.8  0.9
profesional_joven              0.666667  0.8  0.9
pareja_presupuesto_medio       0.333333  0.4  0.3
inversionista_renta_corta      0.666667  0.8  0.9
retirado_tranquilidad          0.000000  0.4  0.7
presupuesto_ajustado_sin_auto  0.333333  0.4  0.2
promedio: {'P3': 0.444, 'P5': 0.6, 'P10': 0.65}

=== estructurado=0.8 / semántico=0.2 ===
                                     P3   P5  P10
perfil                                           
familia_con_ninos              0.666667  0.8  0.7
profesional_joven              0.000000  0.0  0.0
pareja_presupuesto_medio       0.000000  0.2  0.4
inversionista_renta_corta      1.000000  1.0  1.0
retirado_tranquilidad          1.000000  0.8  0.7
presupuesto_ajustado_sin_auto  0.000000  0.4  0.6
promedio: {'P3': 0.444, 'P5': 0.533, 'P10': 0.567}

=== estructurado=

**Resultado decisivo, no ambiguo:** `0.8/0.2`, `0.7/0.3` y `0.6/0.4` (producción) dan
**exactamente el mismo Precision@k, perfil por perfil, en los 3 valores de k** — incluyendo el
mismo 0.000/0.000/0.000 de `profesional_joven` en los tres. Solo `1.0/0.0` (sin componente
semántico) recupera `profesional_joven`, y `0.5/0.5` empeora ligeramente otros perfiles
(`pareja_presupuesto_medio` P@5 baja de 0.2 a 0.0, `inversionista_renta_corta` y
`retirado_tranquilidad` bajan en P@10) sin mejorar `profesional_joven` en absoluto.

**Por qué el peso exacto no importa en el rango 0.2-0.4:** 236 propiedades comparten
`score_estructurado == 1.0` para `profesional_joven` (empate total en el componente estructurado).
Entre propiedades empatadas, **el orden relativo lo decide únicamente el componente semántico** —
y ese orden relativo no cambia con la magnitud del peso semántico, solo con su presencia. Bajar el
peso semántico de 0.4 a 0.2 no cambia qué propiedad gana el desempate, solo qué tan lejos queda del
componente estructurado en la escala absoluta. **No existe un ajuste fino del peso que resuelva el
problema de `profesional_joven` sin eliminar el componente semántico por completo** (`1.0/0.0`) —
la interacción documentada en la sección 9.2 es estructural al método (keywords de exclusión +
embeddings que no las conocen), no sensible al valor exacto del peso.

**Conclusión para la decisión de producción:** no hay evidencia de que mover 0.6/0.4 a 0.7/0.3 u
0.8/0.2 mejore nada — el resultado es idéntico. La disyuntiva real es binaria: mantener el
componente semántico (con el costo ya documentado en `profesional_joven`, y el beneficio ya
documentado en `retirado_tranquilidad`/`inversionista_renta_corta`), o quitarlo del todo.
Mantenemos el scorer híbrido en **0.6/0.4** — el beneficio medible en 2 de 6 perfiles, sostenido
en las 3 métricas, pesa más que el costo documentado y ya explicado en 1 perfil, y no existe una
alternativa de peso intermedio que reduzca ese costo sin renunciar al beneficio.


## 12. Conclusiones

Construimos y evaluamos el componente de matching de M1 de principio a fin: 6 perfiles de
lifestyle sintéticos con ground truth basado en `descripcion` (corregido desde un primer intento
sobre `title` que colapsaba 3 de 6 perfiles a cero), embeddings de `gemini-embedding-001`
(3072 dimensiones) sobre el catálogo completo (1,110 propiedades) y los 6 perfiles, y un scorer
híbrido 0.6 estructurado / 0.4 semántico evaluado con Precision@3/5/10 contra ese ground truth.

El lift promedio del componente semántico sobre un baseline solo-estructurado
(-0.000 / -0.067 / -0.083 para P@3/P@5/P@10) es heterogéneo, no uniforme: aporta lift claro y
positivo en `retirado_tranquilidad` e `inversionista_renta_corta`, y una interacción negativa
específica con `profesional_joven`, ya diagnosticada con evidencia (secciones 8.1, 9.2, 9.3) —
un patrón de mercado real (el copy de marketing rico correlaciona con anuncios de renta corta), no
un defecto del embedding ni de la definición de ground truth por separado. Un sensitivity sweep de
5 combinaciones de peso (sección 10) confirma que ningún ajuste fino del peso resuelve esa
interacción — mantenemos 0.6/0.4 como configuración de producción.

Documentamos la limitación de la regla de exclusión por palabra clave como conocida, con dirección
de trabajo futura explícita (sección 9.3): reemplazarla por una etiqueta estructurada de audiencia
objetivo, candidata natural para el Quality Scorer de 6.2.6 en una iteración posterior — no
implementada en este notebook.

Los artifacts (embeddings de catálogo y perfiles) quedan persistidos en `pipeline/models/`, no en
una ruta temporal, y no dependen de volver a llamar la API para reproducirse.


## 13. Artifacts persistidos

Los embeddings usados en este notebook viven en `pipeline/models/` (convención ya establecida en
Notebook 6.2.1) — `embeddings_catalogo_6_2_3_raw.pkl` y `embeddings_perfiles_6_2_3_raw.pkl` — no en
una ruta temporal. Se generaron una sola vez (sección 6) y se reusan en cada re-ejecución de este
notebook sin volver a llamar la API.
